# Tuần 10: Pedagogical adaptation

Mục tiêu: biến bảng đối chiếu Week 09 thành một lesson adaptation có activity sequence, pacing và limitation.

## Trước khi chạy code

Cell setup là **run-only**: import thư viện, kiểm tra data và tạo output folders.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CANDIDATES = [Path("weeks/week-10-pedagogical-adaptation"), Path(".")]
WEEK_DIR = next(
    candidate for candidate in CANDIDATES
    if (candidate / "data" / "raw" / "week10_pedagogical_adaptation_activities.csv").exists()
    or candidate.name == "week-10-pedagogical-adaptation"
)
DATA_PATH = WEEK_DIR / "data" / "raw" / "week10_pedagogical_adaptation_activities.csv"
TABLE_DIR = WEEK_DIR / "outputs" / "tables"
FIG_DIR = WEEK_DIR / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = "5d1ff3e8d054faddbd2a81fb5cbc17c1ba4f56a3ce37aa29c0921b5939c429c1"
REMOTE_DATA = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-10-pedagogical-adaptation/data/raw/week10_pedagogical_adaptation_activities.csv"
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(REMOTE_DATA, DATA_PATH)
actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Data file:", DATA_PATH)
print("SHA-256:", actual_sha)
if actual_sha != EXPECTED_SHA:
    print("Note: SHA differs from the course snapshot. Continue only if you intentionally changed the data.")


Data file: weeks/week-10-pedagogical-adaptation/data/raw/week10_pedagogical_adaptation_activities.csv
SHA-256: 5d1ff3e8d054faddbd2a81fb5cbc17c1ba4f56a3ce37aa29c0921b5939c429c1


## 1. Load activity data

Một row là một candidate teaching activity. Đây không phải dữ liệu learner thật.

In [2]:
activities = pd.read_csv(DATA_PATH)
activities["include_in_short_lesson"] = activities["include_in_short_lesson"].astype(str).str.lower().eq("true")
print("Candidate activities:", len(activities))
print(activities[["activity_id", "phenomenon", "activity_title", "lesson_phase", "minutes"]].head(8).to_string(index=False))


Candidate activities: 25
activity_id        phenomenon          activity_title           lesson_phase  minutes
       A001 result_complement Notice action vs result                 notice        6
       A002 result_complement     Contrast mini-board                compare        8
       A003 result_complement  Frame completion drill    controlled_practice       10
       A004 result_complement   Lost-ticket role play communicative_practice       10
       A005 result_complement             Exit ticket             assessment        4
       A006     aspect_marker          了 context sort                 notice        8
       A007     aspect_marker   Timeline plus context                compare        8
       A008     aspect_marker   Sentence choice check             assessment        5


## 2. Adaptation score

Score là heuristic lập kế hoạch: ưu tiên Week 09 + difficulty + communicative value - preparation load. Nó giúp chọn activity để thử trước, không chứng minh hiệu quả.

In [3]:
activities["adaptation_score"] = (
    activities["week09_priority_score"]
    + activities["learner_difficulty"] * 2
    + activities["communicative_value"]
    - activities["preparation_load"]
).round(1)
print(activities[["activity_id", "activity_title", "adaptation_score"]].head().to_string(index=False))


activity_id          activity_title  adaptation_score
       A001 Notice action vs result              35.0
       A002     Contrast mini-board              35.0
       A003  Frame completion drill              34.0
       A004   Lost-ticket role play              35.0
       A005             Exit ticket              35.0


## 3. Rank activities với `sort_values()`

`sort_values()` đưa activity có score cao lên trước để ta đọc table dễ hơn.

In [4]:
priority = activities.sort_values(["adaptation_score", "week09_priority_score"], ascending=False).copy()
priority_table = priority[["activity_id", "phenomenon", "activity_title", "lesson_phase", "minutes", "adaptation_score", "activity_goal"]]
priority_table.to_csv(TABLE_DIR / "week10_activity_priority_table.csv", index=False)
print(priority_table.head(10).to_string(index=False))


activity_id        phenomenon              activity_title           lesson_phase  minutes  adaptation_score                                                    activity_goal
       A001 result_complement     Notice action vs result                 notice        6              35.0                                identify action-result difference
       A002 result_complement         Contrast mini-board                compare        8              35.0 compare Chinese result complement with Vietnamese result wording
       A004 result_complement       Lost-ticket role play communicative_practice       10              35.0              use result complements in a tiny communicative task
       A005 result_complement                 Exit ticket             assessment        4              35.0                check whether learner separates action and result
       A003 result_complement      Frame completion drill    controlled_practice       10              34.0                      produc

## 4. Group minutes với `groupby()`

`groupby()` giúp trả lời: lesson plan dành bao nhiêu phút cho từng phase?

In [5]:
phase_order = {
    "review": 1,
    "notice": 2,
    "compare": 3,
    "controlled_practice": 4,
    "communicative_practice": 5,
    "feedback": 6,
    "assessment": 7,
}
plan = activities[activities["include_in_short_lesson"]].copy()
plan["phase_order"] = plan["lesson_phase"].map(phase_order)
plan = plan.sort_values(["phase_order", "activity_id"])
phase_minutes = (
    plan.groupby("lesson_phase", as_index=False)["minutes"].sum()
    .assign(phase_order=lambda d: d["lesson_phase"].map(phase_order))
    .sort_values("phase_order")
    .drop(columns="phase_order")
)
phase_minutes.to_csv(TABLE_DIR / "week10_phase_minutes.csv", index=False)
print(phase_minutes.to_string(index=False))
print("Total lesson minutes:", int(plan["minutes"].sum()))


          lesson_phase  minutes
                review        4
                notice        6
               compare        8
   controlled_practice       10
communicative_practice       10
              feedback        3
            assessment        4
Total lesson minutes: 45


## 5. Short lesson plan table

Bảng lesson plan giữ phase, minutes, teacher move, learner output và assessment evidence.

In [6]:
lesson_plan = plan[["activity_id", "lesson_phase", "minutes", "activity_title", "target_form", "vietnamese_bridge", "teacher_move", "learner_output", "assessment_evidence", "caution_note"]]
lesson_plan.to_csv(TABLE_DIR / "week10_short_lesson_plan.csv", index=False)
print(lesson_plan.to_string(index=False))


activity_id           lesson_phase  minutes          activity_title   target_form              vietnamese_bridge                                          teacher_move             learner_output                         assessment_evidence                                            caution_note
       A025                 review        4      Mini review opener        找 / 找到                 tìm / tìm thấy                              show old and new example       one oral distinction                 learner names action/result                                 Review should be short.
       A001                 notice        6 Notice action vs result        找 / 找到                 tìm / tìm thấy model two examples; ask learners to mark result words        marked example card      learner highlights correct focus token           Do not explain every complement type at once.
       A002                compare        8     Contrast mini-board  找到 / 找完 / 找错 tìm thấy / tìm xong / tìm nhầm     w

## 6. Summary table

Summary table gom thông tin chính để viết paragraph.

In [7]:
summary = pd.DataFrame([
    {"item": "candidate_activities", "value": len(activities)},
    {"item": "selected_activities", "value": len(plan)},
    {"item": "lesson_minutes", "value": int(plan["minutes"].sum())},
    {"item": "top_activity", "value": priority.iloc[0]["activity_title"]},
    {"item": "main_phenomenon", "value": "result_complement"},
])
summary.to_csv(TABLE_DIR / "week10_adaptation_summary.csv", index=False)
print(summary.to_string(index=False))


                item                   value
candidate_activities                      25
 selected_activities                       7
      lesson_minutes                      45
        top_activity Notice action vs result
     main_phenomenon       result_complement


## 7. Figures cho paper

Figure 1 ranking activities. Figure 2 đọc pacing theo phase.

In [8]:
plt.rcParams.update({"font.size": 10, "axes.titlesize": 13, "axes.labelsize": 10})
plot_priority = priority_table.head(10).sort_values("adaptation_score")
fig, ax = plt.subplots(figsize=(8.4, 5.2))
ax.barh(plot_priority["activity_title"], plot_priority["adaptation_score"], color="#2f63ea")
ax.set_title("Week 10 candidate activities ranked by adaptation score")
ax.set_xlabel("Adaptation score")
ax.set_ylabel("Activity")
for i, value in enumerate(plot_priority["adaptation_score"]):
    ax.text(value + 0.2, i, f"{value:.1f}", va="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week10_activity_priority.png", dpi=200)
fig.savefig(FIG_DIR / "week10_activity_priority.svg")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8.2, 4.6))
colors = ["#334155", "#2563eb", "#6d5bd0", "#1f7a4d", "#b45309", "#b8325f", "#64748b"]
ax.bar(phase_minutes["lesson_phase"], phase_minutes["minutes"], color=colors[:len(phase_minutes)])
ax.set_title("Week 10 selected short lesson: minutes by phase")
ax.set_xlabel("Lesson phase")
ax.set_ylabel("Minutes")
ax.tick_params(axis="x", rotation=20)
for i, value in enumerate(phase_minutes["minutes"]):
    ax.text(i, value + 0.3, str(value), ha="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week10_lesson_phase_minutes.png", dpi=200)
fig.savefig(FIG_DIR / "week10_lesson_phase_minutes.svg")
plt.close(fig)
print("Figures exported:")
print(FIG_DIR / "week10_activity_priority.png")
print(FIG_DIR / "week10_lesson_phase_minutes.png")


Figures exported:
weeks/week-10-pedagogical-adaptation/outputs/figures/week10_activity_priority.png
weeks/week-10-pedagogical-adaptation/outputs/figures/week10_lesson_phase_minutes.png


## 8. Pedagogical adaptation paragraph

Dùng table + figure để viết. Điền số liệu sau khi bạn đọc output.

> Based on the activity-priority table, I would adapt the short lesson by focusing on `___`. The lesson begins with `___`, then moves to `___`, and ends with `___`. This recommendation is useful because `___`. However, it remains a design recommendation because the dataset is synthetic and should be tested with real learner responses.

In [9]:
first = lesson_plan.iloc[0]
last = lesson_plan.iloc[-1]
paragraph = (
    f"Based on the activity-priority table, I would adapt a short TCSOL lesson by focusing on result complements. "
    f"The selected plan uses {int(plan['minutes'].sum())} minutes across {len(plan)} activities. "
    f"It begins with {first['activity_title']}, moves through comparison and controlled practice, "
    f"and ends with {last['activity_title']}. This sequence is useful because learners first notice the "
    "action-result contrast before using it in a tiny communicative task. However, the score is only a "
    "planning heuristic from a synthetic dataset, so the adaptation should be tested with real learner responses."
)
print(paragraph)


Based on the activity-priority table, I would adapt a short TCSOL lesson by focusing on result complements. The selected plan uses 45 minutes across 7 activities. It begins with Mini review opener, moves through comparison and controlled practice, and ends with Exit ticket. This sequence is useful because learners first notice the action-result contrast before using it in a tiny communicative task. However, the score is only a planning heuristic from a synthetic dataset, so the adaptation should be tested with real learner responses.
